# BRNS Result Visualization

This notebook visualizes `.dat` output files from a single simulation result directory.
Use it to explore and analyze reaction network simulation results.

**Data format:** Column 0 = concentration/rate, Column 1 = depth

**Workflow:**
1. Configure the result directory path
2. Load all `.dat` files
3. Generate plots (one per file, organized by time snapshots)

For publication, use repository-relative paths over user-specific absolute paths.

In [ ]:
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from notebooks.utils.helper import collect_dat_datasets, split_snapshots

# ============================================================
# Configuration: result directory path
# ============================================================

# TODO: Define the result directory here!
RESULT_DIR = Path('build_output/single_species_example/results')

# When BRNS_RESULT_DIR is set in the environment, use it; otherwise, use the default path
# This allows, that this notebook can be executed in a batch mode or interactively without modifying the code.
RESULT_DIR = Path(
    os.environ.get(
        'BRNS_RESULT_DIR',
        RESULT_DIR, # Default path defined above
    )
)
RESULT_LABEL = os.environ.get('BRNS_RESULT_LABEL', 'BRNS')
COMPARISON_NAME = RESULT_DIR.parent.name

print('Result directory:', RESULT_DIR)
print(f'{RESULT_LABEL:16}:', RESULT_DIR.resolve())
print('Exists          :', RESULT_DIR.exists())
print()

datasets = collect_dat_datasets(RESULT_DIR)

print('.dat files loaded:', len(datasets))
if len(datasets) > 0:
    print('Files:')
    for fname in sorted(datasets.keys()):
        print(f'  - {fname} ({datasets[fname].shape[0]} rows)')

## Result Statistics

Summary of loaded datasets.

In [ ]:
# ============================================================
# Dataset statistics
# ============================================================

print(f"Result Analysis: {RESULT_LABEL}")
print("=" * 60)

for fname, data in sorted(datasets.items()):
    arr = data
    snaps = split_snapshots(arr)
    
    print(f"\n{fname}")
    print(f"  Total rows      : {arr.shape[0]}")
    print(f"  Time snapshots  : {len(snaps)}")
    print(f"  Columns         : {arr.shape[1]}")
    
    if arr.shape[1] >= 2:
        col0_min, col0_max = np.min(arr[:, 0]), np.max(arr[:, 0])
        col1_min, col1_max = np.min(arr[:, 1]), np.max(arr[:, 1])
        print(f"  Col 0 range     : [{col0_min:.3e}, {col0_max:.3e}]")
        print(f"  Col 1 range     : [{col1_min:.3e}, {col1_max:.3e}]")
    
    if len(snaps) > 0:
        print(f"  Rows per snapshot: ", end="")
        print(f"[{', '.join(str(s.shape[0]) for s in snaps[:5])}" + (
            f", ... {snaps[-1].shape[0]}" if len(snaps) > 5 else "") + "]")

## Visualization: One file per figure

Each file is displayed in a separate figure with one subplot per time snapshot.
Time snapshots are detected automatically (depth resets indicate a new time step).

In [ ]:
# ============================================================
# Per file: one figure with ONE column (one subplot per time step)
# Time-step detection: a new step starts when depth resets
# (i.e., when x is smaller than in the previous row)
# ============================================================

for fname, arr in sorted(datasets.items()):
    snaps = split_snapshots(arr)
    n_snaps = len(snaps)

    if n_snaps == 0:
        print(f"⚠ {fname}: no data")
        continue

    print(f"\n=== {fname} ===")
    print(f"Time snapshots: {n_snaps}")

    fig, axes = plt.subplots(
        n_snaps,
        1,
        figsize=(10, max(3.8 * n_snaps, 4.5)),
        squeeze=False
    )
    axes = axes.flatten()

    for i in range(n_snaps):
        ax = axes[i]
        snap = snaps[i]

        # Extract x and y data
        if snap.shape[1] >= 2:
            x, y = snap[:, 1], snap[:, 0]
        else:
            x, y = np.arange(len(snap)), snap[:, 0]

        # Plot
        ax.plot(x, y, '-', color='steelblue', linewidth=2.2, label=RESULT_LABEL)

        ax.set_title(f"{fname} | Time step {i+1}", fontsize=10, fontweight='bold')
        ax.set_xlabel('Depth', fontsize=9)
        ax.set_ylabel('Concentration / Rate', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
        ax.legend(fontsize=8, loc='best')

    fig.suptitle(f"Result visualization: {fname}  [{COMPARISON_NAME}]", fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0.02, 1, 0.97])
    plt.show()

## Optional: Overlay multiple files

Compare different output files in a single plot.

In [ ]:
# ============================================================
# Optional: Overlay different files at the same time snapshot
# Select files and snapshot index to compare
# ============================================================

# Configure which files and snapshot to compare:
FILES_TO_COMPARE = list(sorted(datasets.keys()))[:3]  # First 3 files (adjust as needed)
SNAPSHOT_INDEX = 0  # First time snapshot (adjust as needed)

if len(datasets) > 0 and SNAPSHOT_INDEX >= 0:
    fig, ax = plt.subplots(figsize=(12, 6))

    colors = plt.cm.tab10(np.linspace(0, 1, len(FILES_TO_COMPARE)))

    for idx, fname in enumerate(FILES_TO_COMPARE):
        arr = datasets[fname]
        snaps = split_snapshots(arr)

        if SNAPSHOT_INDEX < len(snaps):
            snap = snaps[SNAPSHOT_INDEX]
            if snap.shape[1] >= 2:
                x, y = snap[:, 1], snap[:, 0]
            else:
                x, y = np.arange(len(snap)), snap[:, 0]

            ax.plot(x, y, '-', linewidth=2.0, label=fname, color=colors[idx])

    ax.set_title(f"Comparison of multiple files | Time step {SNAPSHOT_INDEX + 1}", fontsize=12, fontweight='bold')
    ax.set_xlabel('Depth', fontsize=10)
    ax.set_ylabel('Concentration / Rate', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
    ax.legend(fontsize=9, loc='best')

    plt.tight_layout()
    plt.show()
else:
    print("⚠ No data available for overlay comparison")